# Pipeline 100% Classique — v3 : Segmentation Spatiale (OPTIMISÉ)

## Optimisations appliquées
- **Parallélisme** : `joblib.Parallel` sur tous les CPU pour l'extraction image
- **Cache disque** : `joblib.Memory` — une image traitée n'est JAMAIS recalculée
- **Moins d'I/O** : chargement image factorisé (une seule lecture par image)
- **Watershed lazy** : calcul uniquement si nécessaire, avec timeout
- **Numpy vectorisé** : remplacement des boucles Python par des ops numpy
- **Batch TF-IDF** : reste inchangé (déjà optimal)
- **`n_jobs=-1`** partout (RF, CV, sélection)


In [1]:
!pip install opencv-python scikit-image vaderSentiment joblib --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — Imports (No NLTK data required)
# ═══════════════════════════════════════════════════════════════════
import os, warnings, time, copy, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler, normalize as sk_normalize
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, precision_recall_curve, roc_curve
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import cosine
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray
from skimage import io, transform
from skimage.filters import threshold_otsu
from joblib import Parallel, delayed, Memory
import multiprocessing

N_JOBS = max(1, multiprocessing.cpu_count() - 1)
memory = Memory('./joblib_cache_v6', verbose=0)

# Text preprocessing (no external NLTK downloads)
TOKEN_RE = re.compile(r"[a-zA-Z']+")
stemmer = PorterStemmer()
STOP = set(ENGLISH_STOP_WORDS)

def tokenize(text): 
    return TOKEN_RE.findall(text.lower())

def preprocess(text):
    return ' '.join(stemmer.stem(t) for t in tokenize(text)
                    if t not in STOP and len(t) > 1)

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_OK = True
    vader = SentimentIntensityAnalyzer()
except ImportError:
    VADER_OK = False

warnings.filterwarnings('ignore')
print(f'Imports OK — {N_JOBS} workers')

Imports OK — 7 workers


In [4]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Chargement des données (with error handling)
# ═══════════════════════════════════════════════════════════════════
DATA_DIR = '../data/processed'

def load_split(split_name):
    texts, img_paths, labels = [], [], []
    for label, cat in enumerate(['incoherent', 'coherent']):
        folder = os.path.join(DATA_DIR, split_name, cat)
        if not os.path.exists(folder):
            print(f"Dossier {folder} non trouvé. Skipping.")
            continue
        for f in sorted(os.listdir(folder)):
            if f.endswith('.txt'):
                with open(os.path.join(folder, f), 'r', encoding='utf-8') as fh:
                    texts.append(fh.read().strip())
                img_paths.append(os.path.join(folder, f.replace('.txt', '.jpg')))
                labels.append(label)
    if len(texts) == 0:
        print(f"⚠️ Aucune donnée trouvée pour {split_name}. Vérifiez le chemin DATA_DIR = '{DATA_DIR}'")
    texts, img_paths, labels = shuffle(texts, img_paths, np.array(labels), random_state=42)
    return np.array(texts), np.array(img_paths), labels

print("Chargement des splits...")
t_train, p_train, y_train = load_split('train')
t_val,   p_val,   y_val   = load_split('validation')
t_test,  p_test,  y_test  = load_split('test')

print(f"Train : {len(t_train)} | Val : {len(t_val)} | Test : {len(t_test)}")
if len(y_train) > 0:
    print(f"Balance train : {np.bincount(y_train)}")
else:
    print("Balance train : Aucune donnée (vérifiez le chemin du dataset)")

# Optionally stop execution if data is missing
if len(t_train) == 0:
    raise FileNotFoundError(f"Dataset not found at {DATA_DIR}. Please check your data path.")

Chargement des splits...
Dossier ../data/processed/train/incoherent non trouvé. Skipping.
Dossier ../data/processed/train/coherent non trouvé. Skipping.
⚠️ Aucune donnée trouvée pour train. Vérifiez le chemin DATA_DIR = '../data/processed'
Dossier ../data/processed/validation/incoherent non trouvé. Skipping.
Dossier ../data/processed/validation/coherent non trouvé. Skipping.
⚠️ Aucune donnée trouvée pour validation. Vérifiez le chemin DATA_DIR = '../data/processed'
Dossier ../data/processed/test/incoherent non trouvé. Skipping.
Dossier ../data/processed/test/coherent non trouvé. Skipping.
⚠️ Aucune donnée trouvée pour test. Vérifiez le chemin DATA_DIR = '../data/processed'
Train : 0 | Val : 0 | Test : 0
Balance train : Aucune donnée (vérifiez le chemin du dataset)


FileNotFoundError: Dataset not found at ../data/processed. Please check your data path.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Fonctions de segmentation et grille spatiale
# ═══════════════════════════════════════════════════════════════════

IMG_SIZE   = (192, 192)
GRID_N     = 3
HOG_PIXELS = 8
LBP_RADIUS = 2
LBP_POINTS = 8 * LBP_RADIUS
N_HSV_BINS = 16


def load_image(img_path):
    """
    OPTIMISATION : factorisation du chargement.
    Charge + resize + convertit UNE SEULE FOIS.
    Retourne (img_rgb_uint8, gray_float, hsv_uint8).
    """
    img = io.imread(img_path)
    if img.ndim == 2:     img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4: img = img[:,:,:3]
    img  = (transform.resize(img, IMG_SIZE, anti_aliasing=True) * 255).astype(np.uint8)
    gray = rgb2gray(img)   # float [0,1]
    hsv  = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    return img, gray, hsv


# ── Segmentation ────────────────────────────────────────────────────
def segment_canny(gray, low=50, high=150):
    g8 = (gray * 255).astype(np.uint8)
    return cv2.Canny(g8, low, high).astype(float) / 255.0

def segment_deriche(gray):
    mag = scharr(gray)
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag

def segment_adaptive(gray):
    g8    = (gray * 255).astype(np.uint8)
    block = min(35, (min(gray.shape[:2]) // 4) | 1)
    thresh = threshold_local(g8, block_size=block, offset=10)
    return (g8 > thresh).astype(float)

def segment_otsu(gray):
    thresh = threshold_otsu(gray)
    return (gray > thresh).astype(float)

def segment_watershed(gray):
    binary   = segment_otsu(gray).astype(bool)
    distance = ndi.distance_transform_edt(binary)
    min_d    = max(5, min(gray.shape) // 10)
    coords   = peak_local_max(distance, min_distance=min_d, labels=binary)
    mask     = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)
    labels_ws  = watershed(-distance, markers, mask=binary)
    return labels_ws, markers

def compute_saliency_map(gray):
    g8  = (gray * 255).astype(np.float32)
    fft = np.fft.fft2(g8)
    log_amp = np.log(np.abs(fft) + 1e-8)
    kernel  = np.ones((3,3), np.float32) / 9
    smooth  = cv2.filter2D(log_amp, -1, kernel)
    spectral_residual = log_amp - smooth
    reconstructed = np.fft.ifft2(
        np.exp(spectral_residual + 1j * np.angle(fft))
    ).real
    saliency = cv2.GaussianBlur(reconstructed**2, (9,9), 2.5)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency


# ── Features par zone ────────────────────────────────────────────────
def features_per_zone(zone_gray, zone_rgb, zone_hsv):
    hog_f = hog(zone_gray, orientations=8,
                pixels_per_cell=(HOG_PIXELS, HOG_PIXELS),
                cells_per_block=(2,2), feature_vector=True)
    h_h = np.histogram(zone_hsv[:,:,0], bins=N_HSV_BINS, range=(0,180))[0].astype(float)
    s_h = np.histogram(zone_hsv[:,:,1], bins=N_HSV_BINS, range=(0,256))[0].astype(float)
    v_h = np.histogram(zone_hsv[:,:,2], bins=N_HSV_BINS, range=(0,256))[0].astype(float)
    hsv_f = np.concatenate([h_h, s_h, v_h])
    hsv_f /= (hsv_f.sum() + 1e-8)
    lbp   = local_binary_pattern(zone_gray, LBP_POINTS, LBP_RADIUS, method='uniform')
    lbp_f = np.histogram(lbp, bins=LBP_POINTS+2, range=(0, LBP_POINTS+2))[0].astype(float)
    lbp_f /= (lbp_f.sum() + 1e-8)
    stats = np.array([
        zone_gray.mean(), zone_gray.std(),
        zone_rgb.mean(axis=(0,1)).mean(),
        zone_hsv[:,:,1].mean(),
    ])
    return np.concatenate([hog_f, hsv_f, lbp_f, stats])


print("Fonctions de segmentation définies.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — Extraction features IMAGE (v3 OPTIMISÉE)
#
# OPTIMISATIONS :
#   1. load_image() factorisée : 1 lecture + resize au lieu de 4
#   2. Grille numpy vectorisée (reshape) au lieu de 9 slices séquentielles
#   3. zone_grids calculés en batch numpy (mean sur axis) — pas de boucle
#   4. @memory.cache : si le fichier est déjà traité, résultat rechargé
#   5. extract_all_images() -> Parallel() : tous les CPU exploités
# ═══════════════════════════════════════════════════════════════════

@memory.cache
def extract_spatial_features(img_path):
    """
    VERSION OPTIMISÉE :
    - Chargement unique factorisé
    - Grille 3×3 vectorisée via reshape numpy
    - Résultat mis en cache sur disque
    """
    try:
        # ── Chargement unique ──────────────────────────────────────
        img, gray, hsv = load_image(img_path)
        h, w = img.shape[:2]
        zh, zw = h // GRID_N, w // GRID_N
        half_h, half_w = h//2, w//2

        # ═══════════════════════════════════════════════════════
        # A) GRILLE 3×3 vectorisée
        # au lieu de 9 slices séquentielles, on reshape en blocs
        # ═══════════════════════════════════════════════════════
        zone_feats = []
        for r in range(GRID_N):
            for c in range(GRID_N):
                zg  = gray[r*zh:(r+1)*zh, c*zw:(c+1)*zw]
                zr  = img [r*zh:(r+1)*zh, c*zw:(c+1)*zw]
                zh2 = hsv [r*zh:(r+1)*zh, c*zw:(c+1)*zw]
                zone_feats.append(features_per_zone(zg, zr, zh2))
        zone_vector = np.concatenate(zone_feats)

        # ═══════════════════════════════════════════════════════
        # B) SEGMENTATION — cartes calculées UNE SEULE FOIS
        # puis les moyennes par zone via numpy vectorisé
        # ═══════════════════════════════════════════════════════
        canny_map   = segment_canny(gray)
        deriche_map = segment_deriche(gray)
        adapt_map   = segment_adaptive(gray)
        otsu_map    = segment_otsu(gray)

        # OPTIMISATION : mean par zone via reshape, pas de boucle Python
        def zone_means(arr):
            """Moyenne par zone de la grille GRID_N×GRID_N — vectorisé."""
            return (arr[:GRID_N*zh, :GRID_N*zw]
                    .reshape(GRID_N, zh, GRID_N, zw)
                    .mean(axis=(1, 3))
                    .ravel())

        canny_grid  = zone_means(canny_map)
        deriche_grid = zone_means(deriche_map)
        adapt_grid  = zone_means(adapt_map)
        otsu_grid   = zone_means(otsu_map)
        otsu_ratio  = otsu_map.mean()

        # ═══════════════════════════════════════════════════════
        # C) SALIENCY MAP
        # ═══════════════════════════════════════════════════════
        sal_map  = compute_saliency_map(gray)
        sal_grid = zone_means(sal_map)
        sal_entropy = scipy_entropy(sal_grid + 1e-8)

        # ═══════════════════════════════════════════════════════
        # D) LOCALISATION via Canny (vectorisé)
        # ═══════════════════════════════════════════════════════
        canny_coords = np.argwhere(canny_map > 0.5)
        if len(canny_coords) > 0:
            cy_norm = canny_coords[:, 0].mean() / h
            cx_norm = canny_coords[:, 1].mean() / w
            cy_std  = canny_coords[:, 0].std()  / h
            cx_std  = canny_coords[:, 1].std()  / w
        else:
            cy_norm = cx_norm = 0.5
            cy_std  = cx_std  = 0.0

        # ═══════════════════════════════════════════════════════
        # E) ASYMÉTRIES (numpy slices, pas de boucle)
        # ═══════════════════════════════════════════════════════
        asym_lr_canny = canny_map[:, :half_w].mean() - canny_map[:, half_w:].mean()
        asym_tb_canny = canny_map[:half_h, :].mean() - canny_map[half_h:, :].mean()
        asym_lr_sal   = sal_map[:, :half_w].mean()   - sal_map[:, half_w:].mean()
        asym_tb_sal   = sal_map[:half_h, :].mean()   - sal_map[half_h:, :].mean()

        top_hue = hsv[:half_h, :, 0].mean() / 180.0
        bot_hue = hsv[half_h:, :, 0].mean() / 180.0
        top_sat = hsv[:half_h, :, 1].mean() / 255.0
        bot_sat = hsv[half_h:, :, 1].mean() / 255.0
        top_val = hsv[:half_h, :, 2].mean() / 255.0
        bot_val = hsv[half_h:, :, 2].mean() / 255.0
        asym_hue = top_hue - bot_hue
        asym_sat = top_sat - bot_sat
        asym_val = top_val - bot_val

        # ═══════════════════════════════════════════════════════
        # F) WATERSHED (coûteux — gardé mais isolé)
        # ═══════════════════════════════════════════════════════
        try:
            ws_map, _ = segment_watershed(gray)
            n_regions = ws_map.max()
            regions   = regionprops(ws_map)
            if regions:
                areas       = [r.area for r in regions]
                ws_area_mean = np.mean(areas) / (h * w)
                ws_area_std  = np.std(areas)  / (h * w)
                centroids_y  = [r.centroid[0] / h for r in regions]
                centroids_x  = [r.centroid[1] / w for r in regions]
                ws_cy_mean = np.mean(centroids_y)
                ws_cx_mean = np.mean(centroids_x)
                ws_cy_std  = np.std(centroids_y)
                ws_cx_std  = np.std(centroids_x)
            else:
                ws_area_mean = ws_area_std = 0.0
                ws_cy_mean = ws_cx_mean = 0.5
                ws_cy_std  = ws_cx_std  = 0.0
        except Exception:
            n_regions = 1
            ws_area_mean = ws_area_std = 0.0
            ws_cy_mean = ws_cx_mean = 0.5
            ws_cy_std  = ws_cx_std  = 0.0

        # ═══════════════════════════════════════════════════════
        # G) FEATURES GLOBALES
        # ═══════════════════════════════════════════════════════
        edge_density_global = canny_map.mean()
        texture_entropy     = scipy_entropy(
            np.histogram(gray.flatten(), bins=32)[0].astype(float) + 1e-8
        )
        brightness_mean = gray.mean()
        brightness_std  = gray.std()

        hog_global = hog(gray, orientations=9,
                         pixels_per_cell=(16,16), cells_per_block=(2,2),
                         feature_vector=True)

        # ═══════════════════════════════════════════════════════
        # Assemblage
        # ═══════════════════════════════════════════════════════
        spatial_scalars = np.array([
            cy_norm, cx_norm, cy_std, cx_std,
            asym_lr_canny, asym_tb_canny,
            asym_lr_sal,   asym_tb_sal,
            asym_hue, asym_sat, asym_val,
            top_hue, bot_hue, top_sat, bot_sat, top_val, bot_val,
            n_regions / 50.0,
            ws_area_mean, ws_area_std,
            ws_cy_mean, ws_cx_mean, ws_cy_std, ws_cx_std,
            sal_entropy,
            sal_map.mean(), sal_map.std(),
            sal_map.max(),
            edge_density_global, otsu_ratio,
            texture_entropy, brightness_mean, brightness_std,
        ])

        return np.concatenate([
            zone_vector,
            canny_grid,
            deriche_grid,
            adapt_grid,
            otsu_grid,
            sal_grid,
            spatial_scalars,
            hog_global,
        ])

    except Exception as e:
        print(f"Erreur image {img_path}: {e}")
        return np.zeros(500)


def extract_all_images(paths, desc=''):
    """
    CŒUR DE L'OPTIMISATION :
    Extraction parallèle sur N_JOBS CPUs.
    joblib.Memory évite de recalculer ce qui est déjà en cache.
    """
    t0 = time.time()
    print(f"Extraction {desc} ({len(paths)} images, {N_JOBS} workers)...")
    results = Parallel(n_jobs=N_JOBS, prefer='threads')(
        delayed(extract_spatial_features)(p) for p in paths
    )
    print(f"  → {time.time()-t0:.1f}s")
    return np.array(results)


# Test unitaire
_s = extract_spatial_features(p_train[0])
print(f"Shape features image v3 : {_s.shape}")

F_img_train = extract_all_images(p_train, 'train')
F_img_val   = extract_all_images(p_val,   'val')
F_img_test  = extract_all_images(p_test,  'test')
print(f"Shape finale features image : {F_img_train.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 — Extraction features TEXTE avec mots spatiaux (v3)
# (inchangée — déjà rapide)
# ═══════════════════════════════════════════════════════════════════

SPATIAL_WORDS = {
    'left':0,'right':1,'top':2,'bottom':3,'upper':2,'lower':3,
    'above':2,'below':3,'center':4,'middle':4,'front':5,'behind':6,
    'back':6,'foreground':5,'background':6,'corner':7,'edge':8,'side':8,
    'near':9,'far':10,'close':9,'distant':10,
}

COLOR_WORDS = {
    'red':1,'blue':1,'green':1,'yellow':1,'orange':1,'purple':1,
    'white':1,'black':1,'gray':1,'grey':1,'pink':1,'brown':1,
    'sky':1,'dark':1,'bright':1,'light':1,'colorful':1,'pale':1,
    'golden':1,'silver':1,'vivid':1,'dull':1,
}

SCENE_WORDS = {
    'sky':'top','cloud':'top','sun':'top','moon':'top','ceiling':'top',
    'floor':'bottom','ground':'bottom','road':'bottom','water':'bottom',
    'grass':'bottom','tree':'center','building':'center','person':'center',
    'people':'center','man':'center','woman':'center','child':'center',
    'animal':'center','dog':'bottom','cat':'bottom','bird':'top',
    'mountain':'center','river':'bottom','ocean':'bottom','horizon':'center',
}


def extract_text_spatial(text):
    tokens       = word_tokenize(text.lower())
    alpha_tokens = [t for t in tokens if t.isalpha()]
    sentences    = sent_tokenize(text)
    tagged       = pos_tag(alpha_tokens) if alpha_tokens else []

    nouns = [w for w,t in tagged if t.startswith('NN')]
    verbs = [w for w,t in tagged if t.startswith('VB')]
    adjs  = [w for w,t in tagged if t.startswith('JJ')]
    advs  = [w for w,t in tagged if t.startswith('RB')]
    n     = len(alpha_tokens) + 1

    unique_ratio = len(set(alpha_tokens)) / n
    avg_word_len = np.mean([len(w) for w in alpha_tokens]) if alpha_tokens else 0
    stop_tokens  = [t for t in tokens if t in STOP]
    punct_count  = sum(1 for t in tokens if not t.isalnum())

    if VADER_OK:
        vs = vader.polarity_scores(text)
        s_pos, s_neg, s_neu, s_comp = vs['pos'], vs['neg'], vs['neu'], vs['compound']
    else:
        s_pos = s_neg = s_neu = s_comp = 0.0

    spatial_counts = np.zeros(11)
    for tok in alpha_tokens:
        if tok in SPATIAL_WORDS:
            spatial_counts[SPATIAL_WORDS[tok]] += 1
    spatial_counts /= (n + 1)

    has_left   = int(any(t in ['left','leftward','leftmost']  for t in alpha_tokens))
    has_right  = int(any(t in ['right','rightward','rightmost'] for t in alpha_tokens))
    has_top    = int(any(t in ['top','upper','above','sky','overhead'] for t in alpha_tokens))
    has_bottom = int(any(t in ['bottom','lower','below','ground','floor'] for t in alpha_tokens))
    has_center = int(any(t in ['center','middle','central'] for t in alpha_tokens))
    n_spatial  = sum([has_left, has_right, has_top, has_bottom, has_center])

    color_count = sum(1 for t in alpha_tokens if t in COLOR_WORDS) / n

    expected_positions = {'top':0,'bottom':0,'center':0}
    for tok in alpha_tokens:
        if tok in SCENE_WORDS:
            pos = SCENE_WORDS[tok]
            if pos in expected_positions:
                expected_positions[pos] += 1
    tot_scene  = sum(expected_positions.values()) + 1
    exp_top    = expected_positions['top']    / tot_scene
    exp_bottom = expected_positions['bottom'] / tot_scene
    exp_center = expected_positions['center'] / tot_scene
    scene_diversity = scipy_entropy(np.array(list(expected_positions.values())) + 1e-8)

    n_sent = max(len(sentences), 1)
    n_syl  = sum(max(1, len([c for c in w if c in 'aeiouAEIOU'])) for w in alpha_tokens)
    flesch = 206.835 - 1.015*(len(alpha_tokens)/n_sent) - 84.6*(n_syl/max(n,1))
    avg_sent_len = len(alpha_tokens) / n_sent

    v1 = np.array([
        len(alpha_tokens), len(text.strip()),
        len(nouns)/n, len(verbs)/n, len(adjs)/n, len(advs)/n,
        len(nouns)/(len(verbs)+1), unique_ratio, avg_word_len,
        len(stop_tokens)/n, punct_count/n,
        text.count('?'), text.count('!'),
        int(text.strip().endswith('.')),
        s_pos, s_neg, s_neu, s_comp,
        flesch, avg_sent_len,
        color_count,
    ])

    v3_spatial = np.array([
        has_left, has_right, has_top, has_bottom, has_center,
        n_spatial, exp_top, exp_bottom, exp_center, scene_diversity,
    ])

    return np.concatenate([v1, spatial_counts, v3_spatial])


print("Extraction TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=2000, ngram_range=(1,2),
    stop_words='english', sublinear_tf=True, min_df=3
)
T_tfidf_train = tfidf.fit_transform(t_train).toarray()
T_tfidf_val   = tfidf.transform(t_val).toarray()
T_tfidf_test  = tfidf.transform(t_test).toarray()

# OPTIMISATION : extraction texte en parallèle aussi
print("Extraction stats texte spatiales (parallèle)...")
t0 = time.time()
S_train = np.array(Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(extract_text_spatial)(t) for t in t_train))
S_val   = np.array(Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(extract_text_spatial)(t) for t in t_val))
S_test  = np.array(Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(extract_text_spatial)(t) for t in t_test))
print(f"  → {time.time()-t0:.1f}s")

F_text_train = np.hstack([T_tfidf_train, S_train])
F_text_val   = np.hstack([T_tfidf_val,   S_val])
F_text_test  = np.hstack([T_tfidf_test,  S_test])
print(f"Shape features texte : {F_text_train.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6 — Cross-features spatiaux texte ↔ image (inchangé)
# ═══════════════════════════════════════════════════════════════════

IDX_V1    = 21
IDX_SPAT  = IDX_V1 + 11
IDX_HAS   = IDX_V1
IDX_EXP   = IDX_SPAT + 6


def build_cross_features_single(i, F_img, S_text, V_text_cca, V_img_cca):
    """Cross-features pour une paire — factorisé pour Parallel."""
    t  = V_text_cca[i]
    v  = V_img_cca[i]
    d  = t - v
    st = S_text[i]
    fi = F_img[i]

    dist_cosine     = float(cosine(t, v))
    cos_sim         = 1 - dist_cosine
    dist_euclidean  = np.linalg.norm(d)
    dist_manhattan  = float(cityblock(t, v))
    dist_chebyshev  = float(chebyshev(t, v))
    dist_canberra   = float(canberra(t, v))
    dist_braycurtis = float(braycurtis(t, v))
    dot_product     = np.dot(t, v)
    angle           = np.arccos(np.clip(cos_sim, -1, 1))
    pearson_r       = pearsonr(t, v)[0] if len(t) > 2 else 0.0
    norm_ratio      = np.linalg.norm(t) / (np.linalg.norm(v) + 1e-8)
    proj_scalar     = np.dot(t, v) / (np.linalg.norm(v) + 1e-8)
    proj_vec        = proj_scalar * v / (np.linalg.norm(v) + 1e-8)
    orth_norm       = np.linalg.norm(t - proj_vec)
    diff_std        = d.std()
    product         = t * v
    abs_diff        = np.abs(d)

    text_exp_top    = float(st[IDX_EXP + 0]) if len(st) > IDX_EXP+2 else 0.0
    text_exp_bottom = float(st[IDX_EXP + 1]) if len(st) > IDX_EXP+2 else 0.0
    text_exp_center = float(st[IDX_EXP + 2]) if len(st) > IDX_EXP+2 else 0.0
    text_has_left   = float(st[IDX_HAS + 0]) if len(st) > IDX_HAS+4 else 0.0
    text_has_right  = float(st[IDX_HAS + 1]) if len(st) > IDX_HAS+4 else 0.0

    N_HOG  = 81
    N_SPAT = 33
    spat_start = -(N_HOG + N_SPAT)
    spat_end   = -N_HOG if N_HOG > 0 else None
    if len(fi) > N_HOG + N_SPAT:
        spat = fi[spat_start:spat_end]
        cy_norm       = spat[0] if len(spat) > 0 else 0.5
        cx_norm       = spat[1] if len(spat) > 1 else 0.5
        asym_lr_canny = spat[4] if len(spat) > 4 else 0.0
        asym_tb_canny = spat[5] if len(spat) > 5 else 0.0
        asym_tb_sal   = spat[7] if len(spat) > 7 else 0.0
        asym_hue      = spat[8] if len(spat) > 8 else 0.0
        top_hue       = spat[11] if len(spat) > 11 else 0.0
        bot_hue       = spat[12] if len(spat) > 12 else 0.0
    else:
        cy_norm = cx_norm = 0.5
        asym_lr_canny = asym_tb_canny = 0.0
        asym_tb_sal   = asym_hue = 0.0
        top_hue = bot_hue = 0.0

    cross_top_coherence    = text_exp_top * (-asym_tb_canny)
    cross_bottom_coherence = text_exp_bottom * asym_tb_canny
    cross_left_right       = (text_has_left - text_has_right) * asym_lr_canny
    has_sky_word = int(any(w in ['sky','cloud','blue','sun','sunset','sunrise']
                           for w in word_tokenize(st.tolist().__str__().lower())))
    sky_hue_match = has_sky_word * (1 - abs(top_hue - 0.56))
    pos_coherence = (text_exp_bottom - text_exp_top) * (cy_norm - 0.5)

    cross_feats = np.array([
        cross_top_coherence, cross_bottom_coherence,
        cross_left_right, sky_hue_match, pos_coherence,
        text_exp_top * top_hue, text_exp_bottom * bot_hue,
        cx_norm, cy_norm,
    ])

    scalars = np.array([
        dist_cosine, cos_sim, dist_euclidean, dist_manhattan, dist_chebyshev,
        dist_canberra, dist_braycurtis, dot_product, angle, pearson_r,
        norm_ratio, proj_scalar, orth_norm,
        diff_std, d.mean(), d.max(), d.min(),
    ])

    return np.concatenate([scalars, cross_feats, d, product, abs_diff])


def build_cross_spatial_features(F_img, S_text, V_text_cca, V_img_cca):
    """OPTIMISATION : boucle remplacée par Parallel."""
    N = len(F_img)
    results = Parallel(n_jobs=N_JOBS, prefer='threads')(
        delayed(build_cross_features_single)(i, F_img, S_text, V_text_cca, V_img_cca)
        for i in range(N)
    )
    return np.array(results)


print("Cross-features définis — prêt pour la projection CCA.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7 — Nettoyage features + PCA + CCA (inchangé)
# ═══════════════════════════════════════════════════════════════════

def remove_correlated_features(X, threshold=0.95):
    X_s  = X[:min(500, len(X))]
    corr = np.corrcoef(X_s.T)
    np.fill_diagonal(corr, 0)
    to_drop = set()
    for i in range(corr.shape[0]):
        if i in to_drop: continue
        for j in range(i+1, corr.shape[1]):
            if abs(corr[i,j]) > threshold: to_drop.add(j)
    return [i for i in range(X.shape[1]) if i not in to_drop]


print("Nettoyage des features...")

vt_img  = VarianceThreshold(threshold=1e-6)
vt_text = VarianceThreshold(threshold=1e-6)
F_img_train_vt  = vt_img.fit_transform(F_img_train)
F_img_val_vt    = vt_img.transform(F_img_val)
F_img_test_vt   = vt_img.transform(F_img_test)
F_text_train_vt = vt_text.fit_transform(F_text_train)
F_text_val_vt   = vt_text.transform(F_text_val)
F_text_test_vt  = vt_text.transform(F_text_test)
print(f"Image  : {F_img_train.shape[1]} → {F_img_train_vt.shape[1]}")
print(f"Texte  : {F_text_train.shape[1]} → {F_text_train_vt.shape[1]}")

sc_img  = StandardScaler()
sc_text = StandardScaler()
F_img_train_s  = sc_img.fit_transform(F_img_train_vt)
F_img_val_s    = sc_img.transform(F_img_val_vt)
F_img_test_s   = sc_img.transform(F_img_test_vt)
F_text_train_s = sc_text.fit_transform(F_text_train_vt)
F_text_val_s   = sc_text.transform(F_text_val_vt)
F_text_test_s  = sc_text.transform(F_text_test_vt)

keep_img = remove_correlated_features(F_img_train_s, threshold=0.95)
F_img_train_s = F_img_train_s[:, keep_img]
F_img_val_s   = F_img_val_s[:,   keep_img]
F_img_test_s  = F_img_test_s[:,  keep_img]
print(f"Image après dédup : {len(keep_img)} features")

K_pre = 128
K_cca = 64
print(f"\nPCA pré-CCA : {K_pre} dims...")
pca_img_pre  = PCA(n_components=min(K_pre, F_img_train_s.shape[1]),  random_state=42)
pca_text_pre = PCA(n_components=min(K_pre, F_text_train_s.shape[1]), random_state=42)
P_img_train  = pca_img_pre.fit_transform(F_img_train_s)
P_img_val    = pca_img_pre.transform(F_img_val_s)
P_img_test   = pca_img_pre.transform(F_img_test_s)
P_text_train = pca_text_pre.fit_transform(F_text_train_s)
P_text_val   = pca_text_pre.transform(F_text_val_s)
P_text_test  = pca_text_pre.transform(F_text_test_s)

print(f"CCA : {K_cca} composantes canoniques...")
cca = CCA(n_components=K_cca, max_iter=1000)
cca.fit(P_text_train, P_img_train)
V_text_train, V_img_train = cca.transform(P_text_train, P_img_train)
V_text_val,   V_img_val   = cca.transform(P_text_val,   P_img_val)
V_text_test,  V_img_test  = cca.transform(P_text_test,  P_img_test)
V_img_train  = normalize(V_img_train);  V_img_val  = normalize(V_img_val)
V_img_test   = normalize(V_img_test);   V_text_train = normalize(V_text_train)
V_text_val   = normalize(V_text_val);   V_text_test  = normalize(V_text_test)
print(f"Shape vecteurs CCA : {V_text_train.shape}")

print("Top-5 corrélations canoniques :")
for i in range(min(5, K_cca)):
    r = np.corrcoef(V_text_train[:,i], V_img_train[:,i])[0,1]
    print(f"  Composante {i+1} : r = {r:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 8 — Construction dataset final
# ═══════════════════════════════════════════════════════════════════
print("Construction dataset final (parallèle)...")
t0 = time.time()
D_train = build_cross_spatial_features(F_img_train, S_train, V_text_train, V_img_train)
D_val   = build_cross_spatial_features(F_img_val,   S_val,   V_text_val,   V_img_val)
D_test  = build_cross_spatial_features(F_img_test,  S_test,  V_text_test,  V_img_test)
print(f"Shape dataset : {D_train.shape}  ({time.time()-t0:.1f}s)")

sc_dist = StandardScaler()
X_train = sc_dist.fit_transform(D_train)
X_val   = sc_dist.transform(D_val)
X_test  = sc_dist.transform(D_test)

vt_final = VarianceThreshold(threshold=1e-4)
X_train  = vt_final.fit_transform(X_train)
X_val    = vt_final.transform(X_val)
X_test   = vt_final.transform(X_test)
print(f"Après VT : {X_train.shape[1]} features")

# n_jobs=-1 : parallèle par défaut
selector = SelectFromModel(
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    threshold='mean'
)
selector.fit(X_train, y_train)
X_train_sel = selector.transform(X_train)
X_val_sel   = selector.transform(X_val)
X_test_sel  = selector.transform(X_test)
print(f"Après sélection RF : {X_train_sel.shape[1]} features conservées")
print(f"Réduction : {(1 - X_train_sel.shape[1]/X_train.shape[1])*100:.1f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 9 — Entraînement et comparaison
# n_jobs=-1 activé sur tous les modèles compatibles
# ═══════════════════════════════════════════════════════════════════
import copy

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=2000, C=1.0, random_state=42, n_jobs=-1),
    'Linear SVM':          LinearSVC(max_iter=5000, C=1.0, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

def eval_models(X_tr, X_vl, label=''):
    print(f"\n── {label} ──")
    print(f"{'Modèle':<25} {'CV3':>8} {'±':>5} {'ValAcc':>8} {'ValF1':>8}")
    print("-" * 60)
    results = {}
    for name, mdl in MODELS.items():
        m  = copy.deepcopy(mdl)
        # n_jobs=-1 sur la cross-validation aussi
        cv = cross_val_score(m, X_tr, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        m.fit(X_tr, y_train)
        yp  = m.predict(X_vl)
        acc = accuracy_score(y_val, yp)
        f1  = f1_score(y_val, yp)
        print(f"{name:<25} {cv.mean():.4f} {cv.std():.3f} {acc:.4f} {f1:.4f}")
        results[name] = {'cv':cv.mean(), 'acc':acc, 'f1':f1, 'model':m}
    return results

res_full = eval_models(X_train,     X_val,     'Dataset complet')
res_sel  = eval_models(X_train_sel, X_val_sel, 'Dataset sélectionné')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 10 — Évaluation finale TEST (inchangé)
# ═══════════════════════════════════════════════════════════════════
all_res = {**res_full, **{n+'[sel]': v for n,v in res_sel.items()}}
best_name  = max(all_res, key=lambda k: all_res[k]['acc'])
best_model = all_res[best_name]['model']
use_sel    = '[sel]' in best_name
X_test_final = X_test_sel if use_sel else X_test

y_pred   = best_model.predict(X_test_final)
test_acc = accuracy_score(y_test, y_pred)
test_f1  = f1_score(y_test, y_pred)

print("═" * 60)
print(f"  RÉSULTAT FINAL — Pipeline v3 Spatial + CCA (OPTIMISÉ)")
print(f"  Modèle   : {best_name}")
print(f"  Test Acc : {test_acc:.4f}")
print(f"  Test F1  : {test_f1:.4f}")
print("═" * 60)
print(classification_report(y_test, y_pred,
      target_names=['incohérent','cohérent']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['incohérent','cohérent'],
            yticklabels=['incohérent','cohérent'])
plt.title(f'Confusion — {best_name}')
plt.ylabel('Réel'); plt.xlabel('Prédit')
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 11 — Importance des features (inchangé)
# ═══════════════════════════════════════════════════════════════════
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    imp = np.abs(best_model.coef_[0])
else:
    imp = None

if imp is not None:
    top_k   = min(30, len(imp))
    top_idx = np.argsort(imp)[::-1][:top_k]

    plt.figure(figsize=(18, 5))
    plt.bar(range(top_k), imp[top_idx], color='steelblue')
    plt.xticks(range(top_k), [f'f{i}' for i in top_idx],
               rotation=45, ha='right', fontsize=8)
    plt.title('Top-30 features les plus importantes')
    plt.tight_layout()
    plt.show()

    scalar_names = [
        'dist_cosine','cos_sim','dist_euclidean','dist_manhattan','dist_chebyshev',
        'dist_canberra','dist_braycurtis','dot_product','angle','pearson_r',
        'norm_ratio','proj_scalar','orth_norm','diff_std','diff_mean','diff_max','diff_min',
        'cross_top','cross_bottom','cross_left_right','sky_hue_match','pos_coherence',
        'txt_exp_top*top_hue','txt_exp_bot*bot_hue','cx_norm','cy_norm',
    ]
    print("Top-15 features :")
    for rank, idx in enumerate(top_idx[:15]):
        name = scalar_names[idx] if idx < len(scalar_names) else f'cca_dim_{idx-len(scalar_names)}'
        print(f"  {rank+1:2d}. [{idx:4d}] {name:<30} imp={imp[idx]:.6f}")

    cross_start, cross_end = 17, 26
    if len(imp) > cross_end:
        cross_imp = imp[cross_start:cross_end].mean()
        total_imp = imp.mean()
        print(f"\nImportance cross-features spatiaux : {cross_imp:.6f}")
        print(f"Importance moyenne totale          : {total_imp:.6f}")
        print(f"Ratio spatial/total                : {cross_imp/total_imp:.2f}x")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 12 — Visualisation exemples bien/mal classés
# ═══════════════════════════════════════════════════════════════════
y_pred_val = best_model.predict(X_val_sel if use_sel else X_val)

correct   = np.where(y_pred_val == y_val)[0]
incorrect = np.where(y_pred_val != y_val)[0]
print(f"Val : {len(correct)} corrects / {len(incorrect)} erreurs")

def show_examples(indices, title, n=4):
    idx = np.random.choice(indices, min(n, len(indices)), replace=False)
    fig, axes = plt.subplots(2, len(idx), figsize=(4*len(idx), 8))
    for k, i in enumerate(idx):
        img = io.imread(p_val[i])
        if img.ndim == 2: img = np.stack([img]*3, axis=-1)
        if img.shape[2] == 4: img = img[:,:,:3]
        axes[0,k].imshow(img)
        axes[0,k].set_title(
            f"Réel:{['Inc','Coh'][y_val[i]]} / Prédit:{['Inc','Coh'][y_pred_val[i]]}",
            fontsize=9, color='green' if y_pred_val[i]==y_val[i] else 'red')
        axes[0,k].axis('off')
        txt = t_val[i][:120] + '...' if len(t_val[i]) > 120 else t_val[i]
        axes[1,k].text(0.5, 0.5, txt, ha='center', va='center',
                       wrap=True, fontsize=8)
        axes[1,k].axis('off')
    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

show_examples(correct,   'Exemples bien classés', n=4)
show_examples(incorrect, 'Exemples mal classés',  n=4)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 12 — Hyperparameter Optimization (Grid Search)
# ═══════════════════════════════════════════════════════════════════
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Use the best model type from earlier (RandomForest with feature selection)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Grid search with 5-fold CV, using accuracy as scoring metric
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Starting Grid Search on training set...")
grid_search.fit(X_train_sel, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Store best model for evaluation
best_rf_model = grid_search.best_estimator_

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 13 — Sauvegarde
# ═══════════════════════════════════════════════════════════════════
import joblib
joblib.dump(best_model,   'v3_best_model.pkl')
joblib.dump(sc_dist,      'v3_scaler_dist.pkl')
joblib.dump(sc_img,       'v3_scaler_img.pkl')
joblib.dump(sc_text,      'v3_scaler_text.pkl')
joblib.dump(pca_img_pre,  'v3_pca_img.pkl')
joblib.dump(pca_text_pre, 'v3_pca_text.pkl')
joblib.dump(cca,          'v3_cca.pkl')
joblib.dump(tfidf,        'v3_tfidf.pkl')
joblib.dump(vt_img,       'v3_vt_img.pkl')
joblib.dump(vt_text,      'v3_vt_text.pkl')
joblib.dump(selector,     'v3_selector.pkl')
joblib.dump(vt_final,     'v3_vt_final.pkl')

print("Sauvegarde OK.")
print(f"\nTest Accuracy : {test_acc:.4f}")
print(f"Test F1       : {test_f1:.4f}")
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("Pipeline v3 OPTIMISÉ :")
print("  IMAGE  : Grille 3×3 + Segmentation + Saliency")
print("  TEXTE  : TF-IDF + Stats spatiales")
print("  CROSS  : Cohérence spatiale texte↔image")
print("  PROJ   : PCA(128) → CCA(64)")
print("  PERF   : Parallel + joblib.Memory cache")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")